# Notebook 01 — Country-Year Data (1940–2014)

Builds the country-year (CY) panel that serves as the unit of analysis for both stages of
the *Shadow* model.  The panel covers **1940–2014** (extra pre-period for lags) and is
filtered to the **1946–2014 analysis window** (`inCY`) for modelling.

## What this notebook produces

| Output | Description |
|---|---|
| `country_year.parquet` | Complete CY panel, all variables, all years |
| `cy_imputed_{1..5}.parquet` | Five multiply-imputed copies of the 1946–2014 analysis sample |

The five imputed datasets propagate uncertainty from missing-data imputation through to the
final models.  Notebook 02 cross-joins these into directed-dyad frames, adding five further
UD-level imputations for dyadic auxiliary variables — yielding 25 directed-dyad files in all.

## Design departures from the original paper

The original paper (Carroll 2021) used Fearon and Laitin's (2003) replication data
(`repdata.dta`) directly.  Here every primary variable has been rebuilt from its authoritative
source, with FL's replication data retained only for time-invariant country characteristics
(terrain, colonial history, ethnic fractionalization) that have not been superseded.

| Variable | Original paper | This pipeline |
|---|---|---|
| Civil war onset | Fearon-Laitin (lower threshold, ~25+ deaths) | COW Intra-State Wars v5.1 (≥1,000 battle deaths) |
| Democracy | Polity IV (`polity2`) | V-Dem v15 (`e_polity2` → `polity2`; `v2x_polyarchy` added) |
| GDP per capita | Penn World Tables via F&L | V-Dem `e_gdppc` (2011 PPP international $) |
| Population | NMC `tpop` | V-Dem `e_pop` (NMC v6 `tpop` as backup) |
| Oil wealth | F&L binary | V-Dem `e_total_oil_income_pc` (binarized) |
| Ethnic exclusion | — | EPR 2021 `eth_excl_frac` (new variable) |
| Military capability | NMC v4 | NMC v6 (extends coverage through 2016) |
| Coverage | 1945–1999 | 1940–2014 |

**Threshold note**: the COW ≥1,000-battle-death threshold is substantially higher than F&L's.
The theoretical claim is about great-power intervention decisions, which are triggered by
*major* civil conflicts, making the higher threshold theoretically more appropriate.

## Training / prediction structure

- `inCY` (1946–2014): the analysis window; all CY rows used in stage 1 and stage 2 models
- `regan_period` (1946–1999): subset for which Regan's intervention coding exists;
  the stage 1 classifier is *trained* on this subset and *predicts* for 2000–2014


In [1]:
from __future__ import annotations

import warnings
import zipfile
from pathlib import Path

import miceforest as mf
import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats

from shadow.data.ccode import cc_series, cy_series, fix_ccode

warnings.filterwarnings("ignore", category=FutureWarning)
pd.options.display.max_columns = 60

RAW     = Path("../data/raw")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)

YEAR_START  = 1940   # extra pre-period for lags
YEAR_REGAN  = 1946   # start of Regan intervention training period
YEAR_END    = 2014   # last year with COW v5.1 civil war data

print("Setup complete.")

Setup complete.


## § 1 — State-Year Skeleton (COW State System Members v2011)

The foundation of the CY panel is the **COW State System Membership** list (`states2011.csv`),
which records every state's entry and exit year.  We expand each state's `styear`–`endyear`
interval into individual state-year rows, keeping only years in our analysis window.

States still active at the end of 2011 (the file's last update) are extended through 2014 so
that the panel matches our target coverage.

`fix_ccode` (COW mode) then applies the project's temporal validity rules — dropping rows for
states that were not recognized COW members in a given year (e.g., Austria during Allied
occupation 1945–1954, Germany during the 1945–1948 occupation period, etc.).

**`nwstate`** = 1 in the first two years after a state enters the COW system.  This flag
captures the heightened instability associated with newly independent states and is
used as a covariate in the onset models.

**Note on states list version**: `states2011.csv` is COW v2011; a 2024 update exists but
the 2011 version is consistent with the war and contiguity data already in this pipeline.
The extension to 2014 manually adds three years for states active in 2011.


In [2]:
states = pd.read_csv(RAW / "cow" / "states2011.csv")

# Extend states still active at 2011 through our end year
states.loc[states["endyear"] == 2011, "endyear"] = YEAR_END

rows = []
for _, row in states.iterrows():
    for yr in range(int(row["styear"]), int(row["endyear"]) + 1):
        if YEAR_START <= yr <= YEAR_END:
            rows.append({
                "ccode_raw": int(row["ccode"]),
                "year": yr,
                "statenme": row["statenme"],
                "styear_raw": int(row["styear"]),
            })

cy = pd.DataFrame(rows)

# Apply temporal validity rules
cy["ccode"] = fix_ccode(cc_series(cy["ccode_raw"]), cy["year"])
cy = cy.dropna(subset=["ccode"]).copy()

# Some states have two intervals in states2011 (rejoined COW system);
# deduplicate keeping the more recent entry per (ccode, year)
cy = cy.sort_values("styear_raw", ascending=False).drop_duplicates(
    subset=["ccode", "year"]
).sort_values(["ccode", "year"]).reset_index(drop=True)

# Country name: last (most recent) name per raw ccode
name_map = (
    states.sort_values("styear")
    .groupby("ccode")["statenme"]
    .last()
    .to_dict()
)
cy["cname"] = cy["ccode_raw"].map(name_map)

# New-state indicator: independent within last 2 years
# styear_raw is the start of the current interval
cy["nwstate"] = (cy["year"] - cy["styear_raw"] <= 2).astype(int)

# Analysis sample flags
cy["inCY"]         = cy["year"] >= YEAR_REGAN
cy["regan_period"] = cy["year"] <= 1999

# cyear primary key
cy["cyear"] = cy_series(cy["ccode"], cy["year"])

print(f"State-year rows: {len(cy):,}")
print(f"  inCY (1946–2014): {cy['inCY'].sum():,}")
print(f"  regan_period:     {(cy['inCY'] & cy['regan_period']).sum():,}")
print(f"  Year range: {cy['year'].min()}–{cy['year'].max()}")

State-year rows: 9,236
  inCY (1946–2014): 8,897
  regan_period:     6,516
  Year range: 1940–2014


## § 2 — Civil War Onsets (COW Intra-State Wars v5.1)

**Source**: COW Intra-State War Data v5.1 (`INTRA-STATE WARS v5.1 CSV.csv`), covering
1816–2014.

**War types included**:

| WarType | Label | Description |
|---|---|---|
| 4 | Civil war for central control | Rebels challenge the existing government for control of the state |
| 5 | Civil war for local issues | Rebels seek regional autonomy or secession |

Both types require ≥ **1,000 battle deaths** — the standard COW threshold.  This is a
significantly higher bar than Fearon-Laitin's (~25+ deaths) or UCDP's (25 deaths) thresholds.
The higher threshold is theoretically appropriate here: major interventions by great powers
are unlikely to be triggered by small-scale insurgencies.

**Onset coding**: COW records up to four war *phases* per conflict (some wars restart after
a truce).  We code `onset = 1` in the first year of each phase.  A country can have at most
one onset per year even if two wars began simultaneously.

**Ongoing wars**: we count the number of concurrent active war phases per state-year; this
becomes `ongoing_wars` and feeds the `prior_war` lag.

`CcodeA` in the war file identifies the government-side state.  Rebel-side and external states
(coded in separate columns) are not included in the state-year panel here — their role
as potential interveners is captured in the dyad-level stage.


In [3]:
wars_raw = pd.read_csv(
    RAW / "cow" / "Intra-State-Wars-v5.1" / "INTRA-STATE WARS v5.1 CSV.csv",
    encoding="latin-1",
)

civil = wars_raw[wars_raw["WarType"].isin([4, 5])].copy()

onset_rows, ongoing_rows = [], []

for _, row in civil.iterrows():
    if pd.isna(row["CcodeA"]):
        continue
    raw_cc = int(row["CcodeA"])
    for phase in range(1, 5):
        syr = row.get(f"StartYr{phase}")
        eyr = row.get(f"EndYr{phase}")
        if pd.isna(syr) or int(syr) <= 0:
            continue
        syr = int(syr)
        eyr = int(eyr) if (not pd.isna(eyr) and int(eyr) > 0) else syr
        onset_rows.append({"ccode_raw": raw_cc, "year": syr,
                           "warnum": row["WarNum"]})
        for yr in range(syr, eyr + 1):
            ongoing_rows.append({"ccode_raw": raw_cc, "year": yr,
                                  "warnum": row["WarNum"]})

onset_df   = pd.DataFrame(onset_rows)
ongoing_df = pd.DataFrame(ongoing_rows)

for df_ in (onset_df, ongoing_df):
    df_["ccode"] = fix_ccode(cc_series(df_["ccode_raw"]), df_["year"])

onset_df   = onset_df.dropna(subset=["ccode"])
ongoing_df = ongoing_df.dropna(subset=["ccode"])

# One onset flag per (ccode, year) — a single year can have at most one onset
# per state even if two wars started simultaneously
onset_set = onset_df[["ccode", "year"]].drop_duplicates().copy()
onset_set["onset"] = 1

# Count concurrent ongoing civil wars per (ccode, year)
ongoing_count = (
    ongoing_df.drop_duplicates(subset=["ccode", "year", "warnum"])
    .groupby(["ccode", "year"])
    .size()
    .reset_index(name="ongoing_wars")
)

cy = cy.merge(onset_set,     on=["ccode", "year"], how="left")
cy = cy.merge(ongoing_count, on=["ccode", "year"], how="left")
cy["onset"]        = cy["onset"].fillna(0).astype(int)
cy["ongoing_wars"] = cy["ongoing_wars"].fillna(0).astype(int)

# prior_war: ongoing civil war in previous year (computed here; lagged below)
cy = cy.sort_values(["ccode", "year"])
cy["prior_war"] = (
    cy.groupby("ccode")["ongoing_wars"].shift(1).fillna(0) > 0
).astype(int)

regan_sub = cy[cy["inCY"] & cy["regan_period"]]
print(f"COW v5.1 onsets 1946–1999: {regan_sub['onset'].sum()}")
print(f"  (F&L paper had 111 onsets at lower threshold)")
print(f"COW v5.1 onsets 1946–2014: {cy[cy['inCY']]['onset'].sum()}")

COW v5.1 onsets 1946–1999: 153
  (F&L paper had 111 onsets at lower threshold)
COW v5.1 onsets 1946–2014: 191


## § 3 — V-Dem v15: Democracy and Economic Variables

**Source**: Varieties of Democracy (V-Dem) v15, cross-national time-series 1789–2024.
Loaded directly from the zip archive without extraction (`V-Dem-CY-Full+Others-v15.csv`
inside `V-Dem-CY-FullOthers-v15_csv.zip`; ~400 MB uncompressed).  Only 9 of the ~4,600
columns are read.

| V-Dem column | Derived variable | Notes |
|---|---|---|
| `e_polity2` | `polity2` | Polity II score (−10…10); bridges to original paper |
| `v2x_polyarchy` | — | Electoral democracy index (0…1); modern democracy measure |
| `e_gdppc` | `lgdp` | GDP per capita, 2011 PPP int'l $; log-transformed |
| `e_pop` | `lpop` | Population in thousands; log-transformed |
| `e_total_oil_income_pc` | `oil` | Oil/gas income per capita; binarized (>0) |
| `e_pt_coup` | component of `instab` | Coup d'état indicator (0/1) |
| `v2x_regime` | — | Regime type: closed autocracy / electoral autocracy / electoral democracy / liberal democracy (0–3) |

V-Dem uses COW codes (`COWcode`), so `fix_ccode` harmonises the small number of divergences
(USSR, unified Vietnam, Serbia & Montenegro, etc.).  Where two V-Dem rows map to the same
`(ccode, year)` after recoding, we keep the row with more non-missing values.

**GDP and population**: V-Dem's `e_gdppc` is drawn from the Maddison Project Database and
related sources, expressed in 2011 PPP international dollars.  These supersede the Penn
World Tables series used in the original paper.  `e_pop` is in thousands; log-transformed
`lpop` is used in models.  NMC v6 `tpop` serves as a backup for any rows where V-Dem
population is missing.


In [4]:
VDEM_COLS = [
    "COWcode", "year",
    "e_polity2", "v2x_polyarchy",
    "e_gdppc", "e_pop",
    "e_total_oil_income_pc", "e_pt_coup", "v2x_regime",
]

with zipfile.ZipFile(RAW / "vdem" / "V-Dem-CY-FullOthers-v15_csv.zip") as z:
    with z.open("V-Dem-CY-Full+Others-v15.csv") as f:
        vdem = pd.read_csv(f, usecols=VDEM_COLS, quotechar='"')

vdem = vdem[
    vdem["year"].between(YEAR_START, YEAR_END) &
    vdem["COWcode"].notna()
].copy()

vdem["ccode"] = fix_ccode(cc_series(vdem["COWcode"].astype(int)), vdem["year"])
vdem = vdem.dropna(subset=["ccode"])

# If fix_ccode maps two raw codes to the same (ccode, year), keep the one
# with the most non-null values
vdem["_nv"] = vdem[VDEM_COLS[2:]].notna().sum(axis=1)
vdem = (
    vdem.sort_values("_nv", ascending=False)
    .drop_duplicates(subset=["ccode", "year"])
    .drop(columns=["_nv", "COWcode"])
)

print(f"V-Dem rows loaded: {len(vdem):,}")
print(f"Coverage 1946–2014: {len(vdem[vdem['year'].between(1946,2014)]):,} rows")

V-Dem rows loaded: 10,737
Coverage 1946–2014: 9,967 rows


In [5]:
cy = cy.merge(vdem, on=["ccode", "year"], how="left")

# Derived variables
cy["polity2"] = cy["e_polity2"]
cy["lgdp"]    = np.log(cy["e_gdppc"].clip(lower=1e-6))
cy["lpop"]    = np.log(cy["e_pop"].clip(lower=1e-6))

# Oil: binary if any natural resource income > 0
cy["oil"] = (cy["e_total_oil_income_pc"] > 0).astype(float)
cy.loc[cy["e_total_oil_income_pc"].isna(), "oil"] = np.nan

# Lagged lgdp and lpop
cy = cy.sort_values(["ccode", "year"])
cy["lgdp_lag"] = cy.groupby("ccode")["lgdp"].shift(1)
cy["lpop_lag"] = cy.groupby("ccode")["lpop"].shift(1)

print(f"After V-Dem merge: {len(cy):,} rows")
print("polity2 missingness (inCY):")
print(f"  {cy.loc[cy['inCY'],'polity2'].isna().mean():.1%}")

After V-Dem merge: 9,236 rows
polity2 missingness (inCY):
  1.6%


## § 4 — Polity Lags and Political Instability

Four one-year lags of `polity2` are computed for use as covariates.

**`instab`** (political instability indicator) combines two signals:

1. **Large Polity swing**: |Δpolity2 over a 3-year window| ≥ 3.  A 3-point shift on the
   −10 to +10 scale represents a meaningful change in regime character — consistent with
   the original paper's approach of flagging transitions between authority patterns.
2. **Coup event**: `e_pt_coup == 1` in any year.

Either condition is sufficient to set `instab = 1`.  The variable is set to NaN when
both Polity data and coup data are missing (rather than defaulting to 0).

**Comparison with original paper**: Polity IV codes polity scores as −66 (interregnum),
−77 (transition), −88 (occupation/interruption) for anomalous periods.  The original pipeline
detected instability partly by checking for these special codes.  The V-Dem `e_polity2`
series already maps these to numeric values or NaN, so the original flag-based approach
is replaced by the continuous change criterion above.


In [6]:
cy = cy.sort_values(["ccode", "year"])

for lag in range(1, 5):
    cy[f"polity2_lag{lag}"] = cy.groupby("ccode")["polity2"].shift(lag)

delta3 = (cy["polity2"] - cy["polity2_lag3"]).abs()
coup   = cy["e_pt_coup"].fillna(0).astype(bool)

cy["instab"] = ((delta3 >= 3) | coup).astype(float)
# Where no Polity data AND no coup — leave as NA rather than forcing 0
cy.loc[cy["polity2"].isna() & ~coup, "instab"] = np.nan

instab_rate = cy.loc[cy["inCY"], "instab"].mean()
print(f"Instability rate (inCY 1946–2014): {instab_rate:.3f}")

Instability rate (inCY 1946–2014): 0.139


## § 5 — NMC Capabilities (COW National Material Capabilities v6.0)

**Source**: COW National Material Capabilities (NMC) v6.0 (`NMC_v6_0.csv`), covering
1816–2016.  The file is extracted from a nested zip archive:
`NMC_Documentation-6.0.zip` → `NMC-60-abridged.zip` → `NMC-60-abridged.csv`.

NMC measures six components of national material power, annually:

| Column | Description |
|---|---|
| `irst` | Iron and steel production (thousands of tons) |
| `milex` | Military expenditure (thousands of current-year dollars) |
| `milper` | Military personnel (thousands) |
| `pec` | Primary energy consumption (thousands of coal-ton equivalents) |
| `upop` | Urban population (thousands) |
| `tpop` | Total population (thousands) |
| `cinc` | Composite Index of National Capability (0–1 share of world total) |

`cinc` is the Correlates of War composite capability score, equal to the average of a
state's share of each component across all states in that year.

**Version upgrade**: NMC v4 (used in the original paper) covered 1816–2007.  NMC v6
extends through 2016, eliminating the 2008–2014 gap that would otherwise require
imputation for the extended analysis window.

`tpop` (total population from NMC) provides a backup for `lpop` in the rare cases where
V-Dem `e_pop` is missing.  Missing NMC values are coded −9 in the raw file; these are
treated as NaN.


In [7]:
nmc = pd.read_csv(RAW / "nmc" / "NMC_v6_0.csv", na_values=["-9", ""])

nmc["ccode"] = fix_ccode(cc_series(nmc["ccode"]), nmc["year"])
nmc = nmc.dropna(subset=["ccode"])

NMC_VARS = ["ccode", "year", "irst", "milex", "milper", "pec", "tpop", "upop", "cinc"]
nmc = nmc[NMC_VARS].drop_duplicates(subset=["ccode", "year"])

cy = cy.merge(nmc, on=["ccode", "year"], how="left")

# NMC tpop is in thousands; use as backup for lpop where V-Dem is missing
cy["lpop"] = cy["lpop"].fillna(np.log(cy["tpop"].clip(lower=1e-6)))
cy["lpop_lag"] = cy["lpop_lag"].fillna(
    cy.groupby("ccode")["lpop"].shift(1)
)

print(f"NMC cinc coverage (inCY 1946–2014): {cy.loc[cy['inCY'],'cinc'].notna().mean():.1%}")
print(f"NMC cinc coverage (2008–2014):       {cy.loc[cy['inCY'] & (cy['year'] > 2007),'cinc'].notna().mean():.1%}")

NMC cinc coverage (inCY 1946–2014): 100.0%
NMC cinc coverage (2008–2014):       100.0%


## § 6 — State Power, UN Voting Position, and Intervention History

Four country-year variables capturing each state's strategic standing and foreign policy
orientation.  These serve as predictors in the Stage 1 intervention classifier and, via
the dyad cross-join, as covariates for both sides in Stage 2.

| Variable | Description | Source | Imputed? |
|---|---|---|---|
| `major_power` | COW major power designation (0/1) | COW Major Powers 2024 | No (pass-through) |
| `is_P5` | UN Security Council permanent member (0/1) | Hard-coded from COW codes | No (pass-through) |
| `ideal_point` | UNGA ideal point estimate (liberal ↔ conservative) | Voeten et al., updated Jun 2024 | **Yes** |
| `recent_int` | Count of military interventions by this state in prior 5 years | Regan (2000) | No (NaN post-1999) |

`ideal_point` is the only variable that requires imputation: states absent from UNGA voting
(primarily pre-membership years) have no estimate.  The other three are complete-coverage
indicators treated as pass-through columns in the imputation step.

`recent_int` is derived from the Regan dataset (coverage through 1999).  The 5-year lookback
window becomes incomplete for years after 1999, so post-1999 values are set to NaN and the
variable is used only in the Regan-period sample for Stage 1 training.

In [8]:
from shadow.data.interventions import build_intervention_table

# ── § 6a: Major Power Status (COW Major Powers 2024) ─────────────────────────
majors = pd.read_csv(RAW / "cow" / "MajorPowers2024" / "majors2024.csv")

major_rows = []
for _, row in majors.iterrows():
    for yr in range(int(row["styear"]), int(row["endyear"]) + 1):
        if YEAR_START <= yr <= YEAR_END:
            major_rows.append({"ccode_raw": int(row["ccode"]), "year": yr})

major_df = pd.DataFrame(major_rows)
major_df["ccode"] = fix_ccode(cc_series(major_df["ccode_raw"]), major_df["year"])
major_df = major_df.dropna(subset=["ccode"]).drop_duplicates(subset=["ccode", "year"])
major_df["major_power"] = 1

cy = cy.merge(major_df[["ccode", "year", "major_power"]], on=["ccode", "year"], how="left")
cy["major_power"] = cy["major_power"].fillna(0).astype(int)

# ── § 6b: P5 Status (UN Security Council permanent members) ─────────────────
# COW codes after fix_ccode: USA=002, UK=200, France=220,
#   USSR=364 (through 1991), Russia=365 (from 1992),
#   PRC China=710 (from 1971, when PRC took the UN seat from ROC/Taiwan)
P5_SPECS = [
    ("002", None, None),   # USA: always
    ("200", None, None),   # UK: always
    ("220", None, None),   # France: always in our window (1946+)
    ("364", None, 1991),   # USSR
    ("365", 1992, None),   # Russia
    ("710", 1971, None),   # PRC China
]

def _is_p5(ccode: str, year: int) -> int:
    for cc, yr_from, yr_to in P5_SPECS:
        if ccode != cc:
            continue
        if yr_from is not None and year < yr_from:
            continue
        if yr_to is not None and year > yr_to:
            continue
        return 1
    return 0

cy["is_P5"] = cy.apply(lambda r: _is_p5(r["ccode"], r["year"]), axis=1)

# ── § 6c: UN Ideal Points (Voeten et al., updated Jun 2024) ─────────────────
# IdealpointestimatesAll_Jun2024.csv from Harvard Dataverse
# UNGA session 1 = 1946, so year = session + 1945
with zipfile.ZipFile(RAW / "un-ideals" / "dataverse_files.zip") as z:
    with z.open("IdealpointestimatesAll_Jun2024.csv") as f:
        ip_raw = pd.read_csv(f)

ip = ip_raw[["ccode", "session", "IdealPointAll", "NVotesAll"]].copy()
ip["year"] = ip["session"].astype(int) + 1945
ip = ip[ip["year"].between(YEAR_START, YEAR_END)].copy()
ip["ccode"] = fix_ccode(cc_series(ip["ccode"].astype(int)), ip["year"])
ip = ip.dropna(subset=["ccode"])

# Where fix_ccode maps two raw entries to the same (ccode, year), keep the
# row with the most votes (most reliable estimate).
ip = (
    ip.sort_values("NVotesAll", ascending=False)
    .drop_duplicates(subset=["ccode", "year"])
    .rename(columns={"IdealPointAll": "ideal_point"})
)

cy = cy.merge(ip[["ccode", "year", "ideal_point"]], on=["ccode", "year"], how="left")

# ── § 6d: Recent Intervention Activity (Regan 2000) ─────────────────────────
# Count of distinct military interventions by this state in [year-5, year-1].
# Based on Regan data (coverage 1944–1999); NaN for year > 1999 since the
# 5-year lookback window cannot be fully observed beyond Regan's coverage.
_int_table = build_intervention_table(RAW / "regan" / "replication.10.26.01.dta")

# Index: ccode_B → list of intervention years (one entry per distinct intervention event)
_int_by_B: dict[str, list[int]] = {}
for _, row in _int_table.iterrows():
    _int_by_B.setdefault(str(row["ccode_B"]), []).append(int(row["year"]))

def _recent_int(ccode: str, year: int):
    if year > 1999:
        return np.nan
    years_list = _int_by_B.get(ccode, [])
    return float(sum(1 for y in years_list if year - 5 <= y <= year - 1))

cy["recent_int"] = cy.apply(lambda r: _recent_int(r["ccode"], r["year"]), axis=1)

# ── Summary ──────────────────────────────────────────────────────────────────
incy = cy[cy["inCY"]]
regan_sub = cy[cy["inCY"] & cy["regan_period"]]
major_states = sorted(incy.loc[incy["major_power"] == 1, "ccode"].unique())

print(f"Major power state-years (inCY):  {incy['major_power'].sum():,}")
print(f"  Codes: {major_states}")
print(f"P5 state-years (inCY):           {incy['is_P5'].sum():,}")
print(f"Ideal point coverage (inCY):     {incy['ideal_point'].notna().mean():.1%}")
print(f"Ideal point range:               "
      f"{cy['ideal_point'].min():.2f} – {cy['ideal_point'].max():.2f}")
print(f"Recent interventions (regan sub):"
      f"  mean={regan_sub['recent_int'].mean():.3f}, "
      f"max={regan_sub['recent_int'].max():.0f}")

Major power state-years (inCY):  389
  Codes: ['002', '200', '220', '255', '364', '365', '710', '740']
P5 state-years (inCY):           320
Ideal point coverage (inCY):     92.5%
Ideal point range:               -3.15 – 3.22
Recent interventions (regan sub):  mean=0.306, max=23


## § 6 — Time-Invariant Geography and Colonial Variables (Fearon & Laitin)

**Source**: Fearon and Laitin (2003) replication dataset (`repdata.dta`), accessed via
`pyreadstat`.  This file is used *only* for variables that are effectively time-invariant
and have no readily available replacement: terrain ruggedness, colonial history, and ethnic
fractionalization.

| F&L column | Role |
|---|---|
| `lmtnest` | Log(percent mountainous terrain + 1); terrain ruggedness |
| `ncontig` | Non-contiguous state (islands, exclaves) |
| `colbrit` | Former British colony (binary) |
| `colfra`  | Former French colony (binary) |
| `ethfrac` | Ethnic fractionalization index (0…1; Fearon & Laitin 2003) |
| `relfrac` | Religious fractionalization index (0…1) |
| `muslim`  | Share of population Muslim (0…1; converted from percentage) |
| `region`  | Geographic region (western, eeurope, latinAmerica, mena, africa, asia) |

Because these variables change little or not at all within countries over the sample period,
we take each country's first non-null observation (effectively treating them as constants).
For the F&L dataset, `fix_ccode` is applied in **FL mode** — which uses West Germany (260)
as unified Germany throughout and never includes occupied Germany — to match the original
coding decisions.

**Coverage extension**: States that appear in our panel but not in F&L (post-1999
new states, small island states, etc.) receive region codes from a heuristic based on
COW ccode ranges.  These states still lack terrain and fractionalization values and will
be imputed in § 10.


In [9]:
fl, _ = pyreadstat.read_dta(RAW / "fl" / "repdata.dta")
fl["ccode"] = cc_series(fl["ccode"])
fl["ccode"] = fix_ccode(fl["ccode"], fl["year"], fl_mode=True)
fl = fl.dropna(subset=["ccode"])

TI_VARS = ["ccode", "lmtnest", "ncontig", "colbrit", "colfra",
           "ethfrac", "relfrac", "muslim", "region"]

REGION_MAP = {
    0: "western", 2: "eeurope", 3: "asia",
    5: "mena", 6: "africa", 7: "latinAmerica",
}
fl["region"] = fl["region"].map(REGION_MAP)

# muslim in F&L is percentage; convert to proportion
fl["muslim"] = fl["muslim"] * 0.01

ti = (
    fl[TI_VARS]
    .dropna(subset=["ccode"])
    .sort_values("ccode")
    .groupby("ccode")
    .first()
    .reset_index()
)

cy = cy.merge(ti, on="ccode", how="left")

# ── Extend region to states not in F&L ──────────────────────────────────
# Heuristic based on COW ccode ranges; not perfect but covers most new states.
EEUROPE_CODES = {
    290, 300, 305, 310, 315, 317, 325, 338, 339, 341,
    344, 345, 346, 347, 349, 350, 352, 355, 360, 364,
    366, 367, 368, 369, 370, 371, 372, 373,
}
WESTERN_JAPAN = {740}
WESTERN_OCEANIA = {900, 910, 920, 935, 940, 946, 950, 955, 970}

def region_from_ccode(ccode_str) -> str | None:
    if pd.isna(ccode_str):
        return None
    c = int(ccode_str)
    if c in WESTERN_JAPAN or c in WESTERN_OCEANIA:
        return "western"
    if 1 <= c <= 19:
        return "western"           # US, Canada
    if 20 <= c <= 165:
        return "latinAmerica"
    if 200 <= c <= 395:
        return "eeurope" if c in EEUROPE_CODES else "western"
    if 402 <= c <= 626:
        return "africa"
    if 630 <= c <= 699:
        return "mena"
    if 700 <= c <= 990:
        return "asia"
    return None

missing_reg = cy["region"].isna()
cy.loc[missing_reg, "region"] = cy.loc[missing_reg, "ccode"].apply(region_from_ccode)

print(f"F&L time-invariant coverage: {(~cy['lmtnest'].isna()).mean():.1%}")
print(f"Region assigned:             {cy['region'].notna().mean():.1%}")
print(cy.loc[cy["inCY"], "region"].value_counts())

F&L time-invariant coverage: 99.6%
Region assigned:             100.0%
region
africa          2239
latinAmerica    1535
asia            1464
western         1425
mena            1187
eeurope         1047
Name: count, dtype: int64


## § 7 — EPR 2021: Ethnic Power Relations

**Source**: Ethnic Power Relations (EPR) Core 2021 dataset (`EPR-2021.csv`), covering
1946–2021.  EPR codes the political status and estimated population share of every
politically relevant ethnic group in each country for each year.

**`eth_excl_frac`**: population share of groups coded as **POWERLESS** or
**DISCRIMINATED** in a given country-year.  This is the EPR-based ethnic exclusion
measure developed by Wimmer, Cederman & Min (2009) and captures the degree to which
ethnic minorities are systematically shut out of state power.

EPR uses Gleditsch-Ward (GW) codes (`gwid`), which differ from COW codes for a handful
of cases.  `fix_ccode` handles the main divergences (USSR→364, North Vietnam→818, etc.),
but some GW-only entities (e.g. Palestinian Authority, Kosovo) drop out because they
are not COW members.

**Aggregation**: within each (ccode, year), we sum `size` (population share) for all
groups with excluded status.  The result is clipped to [0, 1].

**Coverage**: EPR data begin in 1946, so pre-1946 rows in the CY panel have
`eth_excl_frac = NaN`; these are handled by multiple imputation in § 10.


In [10]:
epr = pd.read_csv(RAW / "epr" / "EPR-2021.csv")

EXCLUDED_STATUSES = {"POWERLESS", "DISCRIMINATED"}

# Expand group-spell intervals to individual years
epr_rows = []
for _, row in epr.iterrows():
    for yr in range(int(row["from"]), int(row["to"]) + 1):
        if YEAR_START <= yr <= YEAR_END:
            epr_rows.append({
                "gw_ccode": int(row["gwid"]),
                "year": yr,
                "size": float(row["size"]) if pd.notna(row["size"]) else 0.0,
                "excluded": row["status"] in EXCLUDED_STATUSES,
            })

epr_cy = pd.DataFrame(epr_rows)

# Treat gwid as a proxy COW code, then apply fix_ccode to harmonise
epr_cy["ccode"] = fix_ccode(cc_series(epr_cy["gw_ccode"]), epr_cy["year"])
epr_cy = epr_cy.dropna(subset=["ccode"])

# Aggregate: sum excluded population share per (ccode, year)
epr_agg = (
    epr_cy.groupby(["ccode", "year"])
    .apply(
        lambda g: pd.Series({"eth_excl_frac": g.loc[g["excluded"], "size"].sum()}),
        include_groups=False,
    )
    .reset_index()
)
epr_agg["eth_excl_frac"] = epr_agg["eth_excl_frac"].clip(0, 1)

cy = cy.merge(epr_agg, on=["ccode", "year"], how="left")

print(f"EPR coverage (inCY): {cy.loc[cy['inCY'],'eth_excl_frac'].notna().mean():.1%}")
print(f"Mean excluded frac:  {cy.loc[cy['inCY'],'eth_excl_frac'].mean():.3f}")

EPR coverage (inCY): 99.3%
Mean excluded frac:  0.163


## § 8 — Ethnic / Linguistic / Religious Group Labels (Ellingsen 2000)

**Source**: Ellingsen (2000) "witchesbrew" dataset (`witchesbrew1945-2002.dta`), used in
the original paper for constructing same-group indicators at the dyad level.

The dataset assigns each country-year up to two dominant groups along three dimensions:

| Variable | Description |
|---|---|
| `first_eth_grp` / `second_eth_grp` | First and second largest ethnic groups |
| `first_lin_grp` / `second_lin_grp` | First and second largest linguistic groups |
| `first_rel_grp` / `second_rel_grp` | First and second largest religious groups |

Group codes are mapped to readable labels using six lookup text files
(`first_eth_grp.txt`, etc.) provided with the original data.

**Coverage**: the dataset covers 1945–2002.  Because dominant ethnic/linguistic/religious
identities are stable over time, the labels are forward-filled then back-filled within
each country in § 9 — so post-2002 state-years inherit their most recent observed
label, and pre-1945 state-years inherit the earliest.

**Use at the dyad level**: these labels are not used directly as CY covariates.  They
are passed into the dyad cross-join in notebook 02 where they create binary same-group
indicators (e.g., `ud_sameFirstEth = 1` if both states share the same first ethnic group).


In [11]:
eth, _ = pyreadstat.read_dta(RAW / "ethnic" / "witchesbrew1945-2002.dta")
eth = eth[["ccode", "scode", "year",
           "v02d", "v04d", "v07d", "v09d", "v12d", "v14d"]].copy()
eth = eth.rename(columns={
    "v02d": "first_lin_grp",  "v04d": "second_lin_grp",
    "v07d": "first_rel_grp",  "v09d": "second_rel_grp",
    "v12d": "first_eth_grp",  "v14d": "second_eth_grp",
})

eth["ccode"] = cc_series(eth["ccode"])
eth["ccode"] = fix_ccode(eth["ccode"], eth["year"])
eth = eth.dropna(subset=["ccode"])
eth["cyear"] = eth["ccode"] + "_" + eth["year"].astype(int).astype(str)

GROUP_LABEL_COLS = ["first_lin_grp", "second_lin_grp", "first_rel_grp",
                    "second_rel_grp", "first_eth_grp",  "second_eth_grp"]

# Apply lookup tables (numeric code → string label)
for stem in GROUP_LABEL_COLS:
    lut = pd.read_csv(
        RAW / "ethnic" / f"{stem}.txt",
        sep=r"\s+", header=None, names=["code", "label"]
    )
    mapping = dict(zip(lut["code"].astype(float), lut["label"]))
    eth[stem] = eth[stem].map(mapping)   # unmapped → NaN

# Deduplicate: fix_ccode can map multiple raw codes to the same cyear
eth = eth.sort_values("scode").drop_duplicates(subset=["cyear"], keep="first")

eth = eth.drop(columns=["ccode", "scode", "year"])
cy = cy.merge(eth, on="cyear", how="left")

print(f"Group labels merged for {cy['first_eth_grp'].notna().sum():,} / {len(cy):,} rows")

Group labels merged for 6,773 / 9,236 rows


## § 9 — Final Assembly

Collect and order all columns into the `CYdata` frame that will be written to
`country_year.parquet`.

Two housekeeping steps are done here:

1. **Group label forward/back-fill**: `first_eth_grp` and related string columns come from
   the witchesbrew dataset (1945–2002).  Since dominant ethnic/linguistic/religious identity
   categories are stable over time, within-country forward-fill then back-fill extends
   coverage to the full 1940–2014 window with no manual intervention.

2. **Column selection**: `FINAL_COLS` defines the authoritative set of CY variables.  The
   region dummies (`reg_*`) are not included here — they are computed fresh at the start of
   the imputation step (§ 10) from the `region` string column, which is retained in `CYdata`.


In [12]:
GROUP_LABEL_COLS = [
    "first_lin_grp", "second_lin_grp",
    "first_rel_grp",  "second_rel_grp",
    "first_eth_grp",  "second_eth_grp",
]

# Group labels are effectively time-invariant (dominant ethnic/linguistic/
# religious identity rarely changes).  Forward-fill then back-fill within
# each country so that post-2002 and pre-1945 years inherit the nearest
# known value.
cy = cy.sort_values(["ccode", "year"])
for col in GROUP_LABEL_COLS:
    if col in cy.columns:
        cy[col] = cy.groupby("ccode")[col].transform(
            lambda s: s.ffill().bfill()
        )

filled = cy[GROUP_LABEL_COLS[0]].notna().sum()
print(f"Group labels after fill: {filled:,} / {len(cy):,} rows non-null")

Group labels after fill: 8,726 / 9,236 rows non-null


In [13]:
FINAL_COLS = [
    # identifiers
    "cyear", "ccode", "year", "cname",
    # sample flags
    "inCY", "regan_period",
    # civil war outcomes
    "onset", "ongoing_wars", "prior_war",
    # regime
    "polity2", "polity2_lag1", "polity2_lag2", "polity2_lag3", "polity2_lag4",
    "instab", "v2x_polyarchy", "v2x_regime",
    # economic
    "lgdp", "lgdp_lag", "lpop", "lpop_lag", "oil",
    # geography
    "lmtnest", "ncontig", "nwstate",
    # colonial (time-invariant)
    "colbrit", "colfra",
    # ethnic fractionalization (time-invariant, F&L)
    "ethfrac", "relfrac", "muslim",
    # ethnic exclusion (time-varying, EPR)
    "eth_excl_frac",
    # group labels (for dyad stage)
    "first_lin_grp", "second_lin_grp",
    "first_rel_grp",  "second_rel_grp",
    "first_eth_grp",  "second_eth_grp",
    # capabilities (NMC v6)
    "irst", "milex", "milper", "pec", "upop", "cinc",
    # state power and voting position (§ 6)
    "major_power", "is_P5", "ideal_point", "recent_int",
    # region
    "region",
]

FINAL_COLS = [c for c in FINAL_COLS if c in cy.columns]
CYdata = cy[FINAL_COLS].copy()
CYdata["ccode"] = CYdata["ccode"].astype(str)
CYdata = CYdata.sort_values(["ccode", "year"]).reset_index(drop=True)

print(f"CYdata shape: {CYdata.shape}")
print(f"inCY rows:   {CYdata['inCY'].sum():,}  (1946–2014)")
print(f"Regan subset: {(CYdata['inCY'] & CYdata['regan_period']).sum():,}  (1946–1999)")
print(f"Year range:  {CYdata['year'].min()}–{CYdata['year'].max()}")

CYdata shape: (9236, 48)
inCY rows:   8,897  (1946–2014)
Regan subset: 6,516  (1946–1999)
Year range:  1940–2014


## ✓ Validation

Checks performed:

- **No duplicate `cyear` keys**: each country-year appears exactly once.
- **`onset` is binary**: no values outside {0, 1}.
- **`polity2` ∈ [−10, 10]**: V-Dem `e_polity2` should respect the Polity scale bounds.
- **`eth_excl_frac` ∈ [0, 1]**: population share cannot be negative or exceed 1.
- **Onset counts**: compared against the F&L paper (111 onsets 1945–1999) to document the
  expected difference from the COW v5.1 / high-threshold coding decision.
- **Missingness table**: key variables printed to detect any unexpected coverage gaps.


In [14]:
incy   = CYdata[CYdata["inCY"]]
regan  = CYdata[CYdata["inCY"] & CYdata["regan_period"]]

# ── Data integrity ───────────────────────────────────────────────────────
assert CYdata["cyear"].nunique() == len(CYdata), "Duplicate cyear keys!"
assert CYdata["onset"].isin([0, 1]).all(), "onset is not binary"

# ── Polity range ─────────────────────────────────────────────────────────
pol_vals = CYdata["polity2"].dropna()
assert pol_vals.between(-10, 10).all(), (
    f"polity2 out of [-10, 10]: min={pol_vals.min()}, max={pol_vals.max()}"
)

# ── Ethnic exclusion fraction ─────────────────────────────────────────────
excl_vals = CYdata["eth_excl_frac"].dropna()
assert excl_vals.between(0, 1).all(), "eth_excl_frac out of [0, 1]"

# ── COW v5.1 onset benchmarks ─────────────────────────────────────────────
onsets_regan = int(regan["onset"].sum())
onsets_full  = int(incy["onset"].sum())

print(f"COW v5.1 onsets 1946–1999: {onsets_regan}")
print(f"  (F&L paper counted 111 onsets using lower threshold / different coding)")
print(f"COW v5.1 onsets 1946–2014: {onsets_full}")
print()

# ── Key missingness ───────────────────────────────────────────────────────
miss_vars = ["polity2","lgdp","lpop","oil","ethfrac","eth_excl_frac","cinc","lmtnest"]
miss_vars = [v for v in miss_vars if v in incy.columns]
print("Key variable missingness (inCY 1946–2014):")
print(incy[miss_vars].isna().mean().round(3))
print()
print("PASSED all validation checks.")

COW v5.1 onsets 1946–1999: 153
  (F&L paper counted 111 onsets using lower threshold / different coding)
COW v5.1 onsets 1946–2014: 191

Key variable missingness (inCY 1946–2014):
polity2          0.016
lgdp             0.002
lpop             0.000
oil              0.162
ethfrac          0.005
eth_excl_frac    0.007
cinc             0.000
lmtnest          0.005
dtype: float64

PASSED all validation checks.


In [15]:
CYdata.to_parquet(INTERIM / "country_year.parquet", index=False)
print(f"Saved country_year.parquet  ({len(CYdata):,} rows, {CYdata.shape[1]} cols)")

Saved country_year.parquet  (9,236 rows, 48 cols)


## § 10 — Multiple Imputation (miceforest, 5 datasets)

Missing predictors are imputed using **miceforest** (Multiple Imputation by Chained
Equations using random forests / LightGBM).  Only the `inCY` sample (1946–2014) is
imputed.  Five independent datasets are produced, each a complete-case version of the
analysis frame.

**Why multiple imputation?**  Single imputation understates uncertainty by treating
the imputed values as known.  Five datasets allow Rubin's Rules to be applied when
combining model estimates, correctly propagating imputation uncertainty.

**Variables imputed**: all continuous predictors with missing values — polity2, GDP,
population, oil, terrain, ethnic fractionalization, EPR exclusion, and all six NMC
capability measures.  `onset`, `ongoing_wars`, and `prior_war` are included as
predictors (but not imputed) to preserve their relationship with the predictors.

**Asinh pre-transformation**: the six NMC capability variables (`irst`, `milex`,
`milper`, `pec`, `upop`, `cinc`) are right-skewed; they are asinh-transformed
before imputation to approximate normality.  The optimal scale parameter θ is found
by minimising the Kolmogorov-Smirnov statistic between asinh(θ × x) and N(0, 1).
After imputation, the back-transformation `sinh(x) / θ` restores original units,
and the values are clipped to non-negative.

**Post-imputation clipping**: probabilities (`ethfrac`, `relfrac`, `muslim`, etc.) are
clipped to [0, 1]; polity2 is clipped to [−10, 10]; capability variables are clipped
to ≥ 0.

**Deviation from original paper**: the original paper used Amelia II (multiple imputation
via the EM algorithm with bootstrapping).  miceforest is a better choice for this
pipeline because it handles non-normal distributions more flexibly via random forests
and natively supports predictor constraints.

**Output**: `cy_imputed_1.parquet` through `cy_imputed_5.parquet`, each containing the
full `inCY` sample with all numeric variables complete.  The `cyear` and `ccode`
identifiers are reattached so that downstream notebooks can join back on these keys.


In [16]:
# Variables to include in imputation frame
IMPUTE_VARS = [
    "onset", "ongoing_wars", "prior_war",
    "polity2", "instab", "v2x_polyarchy",
    "lgdp", "lgdp_lag", "lpop", "lpop_lag", "oil",
    "lmtnest", "ncontig", "nwstate",
    "colbrit", "colfra", "ethfrac", "relfrac", "muslim",
    "eth_excl_frac",
    "irst", "milex", "milper", "pec", "upop", "cinc",
    "ideal_point",   # § 6c: genuine missingness; imputed via miceforest
    "year",          # kept as predictor for temporal structure
]
IMPUTE_VARS = [v for v in IMPUTE_VARS if v in CYdata.columns]

# Region dummies (include group structure as predictors)
region_dummies = pd.get_dummies(
    CYdata.loc[CYdata["inCY"], "region"].reset_index(drop=True),
    prefix="reg", drop_first=False
)
# Drop one reference category
for ref in ["reg_africa"]:
    if ref in region_dummies.columns:
        region_dummies = region_dummies.drop(columns=ref)

incy_idx = CYdata["inCY"]
cy_imp = pd.concat(
    [CYdata.loc[incy_idx, IMPUTE_VARS].reset_index(drop=True),
     region_dummies],
    axis=1
)

print(f"Imputation frame: {cy_imp.shape}")
print(f"Columns with missing data: {cy_imp.isna().any().sum()}")

Imputation frame: (8897, 33)
Columns with missing data: 19


In [17]:
# Inverse hyperbolic sine normalisation for skewed NMC variables
SINH_VARS = ["irst", "milex", "milper", "pec", "upop", "cinc"]
SINH_VARS = [v for v in SINH_VARS if v in cy_imp.columns]

def ks_stat_vs_normal(x: np.ndarray) -> float:
    x = x[~np.isnan(x)]
    if len(x) < 20:
        return 1.0
    x_std = (x - x.mean()) / (x.std() + 1e-12)
    stat, _ = stats.kstest(x_std, "norm")
    return float(stat)

theta_candidates = 2.0 ** np.arange(-20, 21, dtype=float)
theta_map = {}

np.random.seed(90210)
for name in SINH_VARS:
    x = cy_imp[name].to_numpy(dtype=float)
    losses = [ks_stat_vs_normal(np.arcsinh(t * x)) for t in theta_candidates]
    best_t = theta_candidates[int(np.argmin(losses))]
    theta_map[name] = best_t
    cy_imp[name] = np.arcsinh(best_t * cy_imp[name])

print("Optimal asinh θ per NMC variable:")
for k, v in theta_map.items():
    print(f"  {k:<10} 2^{np.log2(v):.0f}")

Optimal asinh θ per NMC variable:
  irst       2^-6
  milex      2^-7
  milper     2^1
  pec        2^-8
  upop       2^-7
  cinc       2^20


In [18]:
# miceforest requires all-numeric float64
cy_imp = cy_imp.apply(pd.to_numeric, errors="coerce").astype("float64")
assert all(cy_imp.dtypes == "float64"), "Non-float columns remain!"

print(f"Total missing cells: {cy_imp.isna().sum().sum():,}")

Total missing cells: 3,619


In [19]:
NUM_IMP = 5

kernel = mf.ImputationKernel(
    data=cy_imp,
    num_datasets=NUM_IMP,
    random_state=90210,
)
kernel.mice(5, verbose=False)
print(f"miceforest: {NUM_IMP} imputations complete.")

miceforest: 5 imputations complete.


In [20]:
# Save imputed datasets with identifiers reattached.
# Pass-through columns (major_power, is_P5, recent_int) are taken directly
# from CYdata and do not go through miceforest.  ideal_point IS imputed and
# comes back via kernel.complete_data(i).
id_cols = CYdata.loc[incy_idx, [
    "cyear", "ccode", "year", "cname", "inCY", "regan_period",
    "major_power", "is_P5", "recent_int",
]].reset_index(drop=True)

for i in range(NUM_IMP):
    imp = kernel.complete_data(i)   # 0-indexed

    # Post-imputation clipping to valid bounds
    clips = {
        "polity2":       (-10, 10),
        "ethfrac":       (0, 1),
        "relfrac":       (0, 1),
        "muslim":        (0, 1),
        "eth_excl_frac": (0, 1),
        "v2x_polyarchy": (0, 1),
        "instab":        (0, 1),
        "ongoing_wars":  (0, None),
        "cinc":          (0, None),
    }
    for col, (lo, hi) in clips.items():
        if col in imp.columns:
            imp[col] = imp[col].clip(lower=lo, upper=hi)

    # Back-transform sinh variables
    for name, t in theta_map.items():
        if name in imp.columns:
            imp[name] = np.sinh(imp[name].to_numpy(dtype=float)) / t

    # Drop columns that are already in id_cols to avoid duplicates
    imp_clean = imp.drop(columns=["year"], errors="ignore").reset_index(drop=True)
    out = pd.concat([id_cols, imp_clean], axis=1)
    out.to_parquet(INTERIM / f"cy_imputed_{i+1}.parquet", index=False)

print(f"Saved cy_imputed_1.parquet … cy_imputed_{NUM_IMP}.parquet")

# Verify: no missing numeric values
for i in range(1, NUM_IMP + 1):
    imp = pd.read_parquet(INTERIM / f"cy_imputed_{i}.parquet")
    n_miss = imp.select_dtypes(include=np.number).isna().sum().sum()
    print(f"  cy_imputed_{i}: {n_miss} missing numeric values")

Saved cy_imputed_1.parquet … cy_imputed_5.parquet


  cy_imputed_1: 2381 missing numeric values
  cy_imputed_2: 2381 missing numeric values
  cy_imputed_3: 2381 missing numeric values
  cy_imputed_4: 2381 missing numeric values
  cy_imputed_5: 2381 missing numeric values
